In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load all datasets
listings_core = pd.read_csv('../data/listings.csv.gz', compression='gzip')
listings_detailed = pd.read_csv('../data/listings.csv')
calendar = pd.read_csv('../data/calendar.csv.gz', compression='gzip')
reviews = pd.read_csv('../data/reviews.csv')
neighbourhoods = pd.read_csv('../data/neighbourhoods.csv')

print("All datasets loaded successfully!")
print(f"\nlistings_core:     {listings_core.shape}")
print(f"listings_detailed: {listings_detailed.shape}")
print(f"calendar:          {calendar.shape}")
print(f"reviews:           {reviews.shape}")
print(f"neighbourhoods:    {neighbourhoods.shape}")

In [ ]:
# Explore listings_core columns and types
print("=" * 60)
print("LISTINGS CORE - Column Info")
print("=" * 60)
print(listings_core.dtypes)

print("\n" + "=" * 60)
print("LISTINGS CORE - First 3 rows (key columns only)")
print("=" * 60)
key_cols = ['id', 'name', 'host_id', 'host_name', 'neighbourhood', 
            'latitude', 'longitude', 'room_type', 'price', 
            'minimum_nights', 'number_of_reviews', 'availability_365']
print(listings_core[key_cols].head(3))

In [ ]:
# Explore calendar
print("=" * 60)
print("CALENDAR - Column Info")
print("=" * 60)
print(calendar.dtypes)
print("\nFirst 3 rows:")
print(calendar.head(3))

print("\n" + "=" * 60)
print("REVIEWS - Column Info")
print("=" * 60)
print(reviews.dtypes)
print("\nFirst 3 rows:")
print(reviews.head(3))

print("\n" + "=" * 60)
print("NEIGHBOURHOODS - Column Info")
print("=" * 60)
print(neighbourhoods.dtypes)
print("\nAll rows (only 50):")
print(neighbourhoods.head(10))

In [ ]:
# Missing value analysis
def missing_value_report(df, name):
    print(f"\n{'=' * 60}")
    print(f"MISSING VALUES - {name}")
    print(f"{'=' * 60}")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    report = pd.DataFrame({
        'Missing Count': missing,
        'Missing %': missing_pct
    })
    report = report[report['Missing Count'] > 0].sort_values('Missing %', ascending=False)
    if len(report) == 0:
        print("No missing values!")
    else:
        print(report.to_string())

missing_value_report(listings_core, "Listings Core")
missing_value_report(calendar, "Calendar")
missing_value_report(reviews, "Reviews")
missing_value_report(neighbourhoods, "Neighbourhoods")

In [ ]:
# Relationship mapping between datasets
print("=" * 60)
print("KEY RELATIONSHIPS BETWEEN DATASETS")
print("=" * 60)

# listings <-> calendar
listing_ids = set(listings_core['id'])
calendar_ids = set(calendar['listing_id'])
review_ids = set(reviews['listing_id'])

print(f"\nTotal unique listings:                    {len(listing_ids)}")
print(f"Listings in calendar:                     {len(calendar_ids)}")
print(f"Listings in reviews:                      {len(review_ids)}")

print(f"\nCalendar IDs matching listings:           {len(calendar_ids & listing_ids)}")
print(f"Calendar IDs NOT in listings:             {len(calendar_ids - listing_ids)}")

print(f"\nReview IDs matching listings:             {len(review_ids & listing_ids)}")
print(f"Review IDs NOT in listings:               {len(review_ids - listing_ids)}")

print(f"\nListings with NO calendar entries:        {len(listing_ids - calendar_ids)}")
print(f"Listings with NO reviews:                 {len(listing_ids - review_ids)}")

In [ ]:
# Data dictionary - key columns analysis
print("=" * 60)
print("ROOM TYPES")
print("=" * 60)
print(listings_core['room_type'].value_counts())

print("\n" + "=" * 60)
print("TOP 10 NEIGHBOURHOODS")
print("=" * 60)
print(listings_core['neighbourhood_cleansed'].value_counts().head(10))

print("\n" + "=" * 60)
print("PRICE ANALYSIS (raw)")
print("=" * 60)
print(listings_core['price'].describe())

print("\n" + "=" * 60)
print("AVAILABILITY STATS")
print("=" * 60)
print(listings_core['availability_365'].describe())

print("\n" + "=" * 60)
print("MINIMUM NIGHTS")
print("=" * 60)
print(listings_core['minimum_nights'].describe())

print("\n" + "=" * 60)
print("HOST STATS")
print("=" * 60)
print(f"Total unique hosts: {listings_core['host_id'].nunique()}")
print(f"Superhost breakdown:")
print(listings_core['host_is_superhost'].value_counts())

In [ ]:
# Dataset limitations analysis
print("=" * 60)
print("DATASET LIMITATIONS")
print("=" * 60)

# 1. Scraping date
print(f"\n1. Data scrape date:")
print(listings_core['last_scraped'].value_counts())

# 2. Outlier check - minimum nights
print(f"\n2. Suspicious minimum_nights values (>365):")
print(listings_core[listings_core['minimum_nights'] > 365][['id','name','minimum_nights']].head(5))

# 3. Zero availability listings
zero_avail = listings_core[listings_core['availability_365'] == 0]
print(f"\n3. Listings with 0 availability: {len(zero_avail)}")
print(f"   These may be inactive or fully booked listings")

# 4. Price outliers
print(f"\n4. Listings with missing price: {listings_core['price'].isnull().sum()}")

# 5. Calendar price issue
print(f"\n5. Calendar price columns are 100% null")
print(f"   Revenue estimation must rely on listings.price instead")

# 6. Review score coverage
print(f"\n6. Listings with review scores: {listings_core['review_scores_rating'].notna().sum()}")
print(f"   Listings without review scores: {listings_core['review_scores_rating'].isna().sum()}")
print(f"   Reason: new listings with 0 reviews get no score")

In [ ]:
with open('../reports/section02_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary)

print("Section 02 summary saved to reports/section02_summary.txt")
print("Section 02 - Dataset Familiarization COMPLETE!")